### Load Multiple-Choice Questions

In [ ]:
import json

with open("data/ptpt-multiple-choice-qa-pairs.json", "r", encoding="utf-8") as f:
    mcq_items = json.load(f)

### Expand references for some ptpt qa pairs

In [ ]:
import re
from typing import Dict, List

def expand_references_in_question(
	question: str,
	images: Dict[str, str] | None,
	tables: Dict[str, str] | None,
) -> str:
	images = images or {}
	tables = tables or {}

	# Find tokens in parentheses like (image1), (table1), etc.
	tokens = re.findall(r"\(([^)]+)\)", question or "")
	# Preserve first-seen order without duplicates
	seen: set[str] = set()
	references: List[str] = []
	for t in tokens:
		if t in seen:
			continue
		if t in images or t in tables:
			seen.add(t)
			references.append(t)

	if not references:
		return question

	parts = [question, "\n\nReferenced content:"]
	for key in references:
		parts.append(f"[{key}]\n{images.get(key) if key in images else tables.get(key, '')}")
	return "\n".join(parts)

### Add prompt instructions and format options

In [ ]:
def build_prompt(question_text: str, options_text: Dict[str, str]) -> str:
	header = (
		"Solve the following math multiple-choice question. Make sure to put the correct option letter (A,B,C,D or E), and only the correct option letter, inside \\boxed{}.\n\n"
	)
	# Ensure options appear ordered as A..E (only those present)
	ordered_keys = [k for k in ["A", "B", "C", "D", "E"] if k in options_text]
	options_lines = "\n".join(f"{k}) {options_text[k]}" for k in ordered_keys)
	prompt = f"{header}Question: {question_text}\n\n{options_lines}\n"
	return prompt

#### Final Result Example:
<pre>
Solve the following math multiple-choice question. Make sure to put the correct option letter (A,B,C,D or E), and only the correct option letter, inside \boxed{}.

Question: Nos quadrados da figura seguinte, os vértices $A,\,B$ e $C$ estão sobre uma recta: (image1) O valor de $x$ é igual a


Referenced content:
[image1]
\begin{picture}(121.00,75.00)(0,10) \put(5.00,5.00){\line(1,0){20.00}} \put(25.00,5.00){\line(0,1){20.00}} \put(25.00,25.00){\line(-1,0){20.00}} \put(5.00,25.00){\line(0,-1){20.00}} \put(25.00,5.00){\line(1,0){35.00}} \put(60.00,5.00){\line(0,1){35.00}} \put(60.00,40.00){\line(-1,0){35.00}} \put(25.00,40.00){\line(0,-1){35.00}} \put(121.00,5.00){\line(0,1){61.00}} \put(121.00,66.00){\line(-1,0){61.00}} \put(60.00,66.00){\line(0,-1){61.00}} \put(60.00,5.00){\line(1,0){61.00}} \put(15.00,8.00){\makebox(0,0)[cb]{$4$}} \put(42.00,8.00){\makebox(0,0)[cb]{$7$}} \put(90.00,8.00){\makebox(0,0)[cb]{$x$}} \put(3.00,27.00){\makebox(0,0)[rb]{$A$}} \put(24.00,41.00){\makebox(0,0)[rb]{$B$}} \put(59.00,67.00){\makebox(0,0)[rb]{$C$}} \put(8.50,22.50){\makebox(0,0)[rb]{$\bullet$}} \put(28.50,37.50){\makebox(0,0)[rb]{$\bullet$}} \put(63.00,63.00){\makebox(0,0)[rb]{$\bullet$}} \end{picture}

A) 10
B) $\frac{49}{4}$
C) 11
D) $\frac{33}{4}$
E) $\frac{77}{4}$
</pre>

### Loop that creates a prompt for every item in the input .json file

In [ ]:
prompts = []

for item in mcq_items:
    if not item.get("multiple_choice_question", False):
        continue
    question = item.get("question_verbatim")
    options = item.get("options_verbatim")
    if not question or not options:
        print(f"Skipping item without valid question/options: {item.get('id', 'unknown id')}")
        continue
    expanded_question = expand_references_in_question(
			question,
			item.get("images"),
			item.get("tables"),
	)
    prompt_text = build_prompt(expanded_question, options)
    prompts.append({
        "id": item.get("id"),
        "prompt": prompt_text,
    })

### Print prompts

In [ ]:
for entry in prompts:
    # Pretty-print with id context
    pid = entry.get("id", "unknown")
    print(f"id={pid}\n{entry.get('prompt','')}\n")

### Store created prompts in a .json file

In [ ]:
with open("prompts/ptbr-prompts.json", "w", encoding="utf-8") as f:
    json.dump(prompts, f, ensure_ascii=False, indent=2)